## Downloading Necessary Libraries & Packages

In [ ]:
!pip install trimesh

## Importing Necessary Libraries, Packages and Functions

In [ ]:
import trimesh
import numpy as np
import nibabel as nib
from scipy.interpolate import RegularGridInterpolator
from CEHE_Algorithm import get_density_map, CEHE, inverted_average_fn
from scipy.ndimage import map_coordinates

## Some Utility Functions:
These functions should be implemented in AOMT and the algorithm implemented here should be supplied by vertices coordinates of the mesh, faces of the mesh, and density of each vertex.

In [ ]:
def build_density_map(file_path):
  img = nib.load(file_path)
  img_arr = img.get_fdata()
  normalized_img_arr = (img_arr - np.mean(img_arr)) / np.std(img_arr)
  enhanced_img_arr = CEHE(normalized_img_arr, inverted_average_fn, 3, 65536)
  enhanced_image = nib.Nifti1Image(enhanced_img_arr, affine=np.eye(4))
  return get_density_map(enhanced_image, 1)

In [ ]:
def read_off_file(file_path):
  mesh = trimesh.load_mesh(file_path)
  return np.asarray(mesh.vertices), np.asarray(mesh.faces)

In [ ]:
d_map = build_density_map("/content/BraTS2021_00000_flair.nii.gz")
vertices, faces = read_off_file('/content/BraTS2021_Training_00000_flair.off')

## A Function to compute vertices density
It performs Trilinear Interpolation on each vertex coordinate to get its density from the image density map.

In [ ]:
get_vertices_density = lambda d_map, vertices: map_coordinates(d_map, vertices.T, order=1, mode='nearest')

## Building Edge Augmented Matrix

### a Function to build Adjacency Matrix for the Mesh

In [ ]:
def build_adj_matrix(mesh_faces, num_vertices):
  adj_matrix = np.zeros((num_vertices, num_vertices)).astype(bool)

  def fill_adj_matrix(face):
    v1, v2, v3 = face[0], face[1], face[2]
    adj_matrix[v1, v2], adj_matrix[v1, v3], adj_matrix[v2, v3] = True, True, True
    adj_matrix[v2, v1], adj_matrix[v3, v1], adj_matrix[v3, v2] = True, True, True

  np.apply_along_axis(fill_adj_matrix, axis=1, arr=mesh_faces)
  return adj_matrix

In [ ]:
def ASEM(mesh_vertices, mesh_faces, density_map):
  vertices = get_vertices_density(density_map, vertices)
  adj_matrix = build_adj_matrix(np.array([[1, 2, 3], [2, 3, 4], [3, 4, 5]]), 6)